# Label Comparison Notebook

Compares grid-search runs stored under **different labels** (e.g. `before-fix` vs `after-fix`)
to evaluate the effect of topology or implementation changes.

Each label maps to one run group produced by `run-grid-search.py --label <name>`.
Runs inside each group can span multiple timestamps (grid batches) — all are loaded and aggregated.

**Metrics compared:**
- Traffic compliance rate & violations
- Tick tuple irregularity (normalized mean/std/max deviation)
- Throughput (tuples/s and tuples/epoch)
- Snapshot duration (enclave compute cost)
- Epoch duration
- Real vs dummy emission ratio

In [ ]:
from pathlib import Path
from typing import cast

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import pandas as pd
from IPython.display import display

from helpers.dataset import (
    discover_runs,
    detect_varying_params,
    short_label,
    compact_number,
    build_comparison_df,
    load_profiler_csvs,
    compute_run_summary,
)
from helpers.visualization import (
    save_or_show,
    color_for,
    _PARAM_DISPLAY,
)

## Configuration

Set `LABEL_DIRS` to a list of `(label_name, path_to_label_dir)` pairs.
Each path should be the label subdirectory directly (e.g. `data/runs/before-fix/`),
which contains one or more `<timestamp>/` subdirectories of runs.

```
data/runs/
  before-fix/
    20260330_120000/
      tick90_epochs50_p8_mu100_u1000000_k1000000_run1/
        params.txt
        profiler/
  after-fix/
    20260331_090000/
      tick90_epochs50_p8_mu100_u1000000_k1000000_run1/
        ...
```

In [ ]:
ENCLAVE_RUNS_ROOT = Path("../examples/synthetic-dp-histogram/data/runs")

# --- List of (display_label, path_to_label_dir) ---
# Each path is the label subdirectory under data/runs/
LABEL_DIRS: list[tuple[str, Path]] = [
    ("with-lock", ENCLAVE_RUNS_ROOT / "with-locking"),
    ("without-lock",  ENCLAVE_RUNS_ROOT / "without-locking"),
]

# --- Filters ---
# Set to a specific topology_type to restrict (e.g. "enclave" or "baseline").
# None = include all.
FILTER_TOPOLOGY_TYPE = "enclave"

# Minimum epochs completed to include a run in traffic analysis
# (early epochs show variability; 0 = include all)
MIN_EPOCHS = 0

# --- Output ---
OUTPUT_DIR = Path("plots/label-comparison")
SAVE_PLOTS = False
PLOT_FORMAT = "png"

if SAVE_PLOTS:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

## Load Runs

In [ ]:
frames: list[pd.DataFrame] = []

for display_label, label_path in LABEL_DIRS:
    if not label_path.is_dir():
        print(f"WARNING: label directory not found, skipping: {label_path}")
        continue
    try:
        df = discover_runs(label_path)
        df["run_label"] = display_label  # our comparison grouping column
        frames.append(df)
        print(f"  [{display_label}] {len(df)} runs from {label_path}")
    except (FileNotFoundError, ValueError) as exc:
        print(f"  [{display_label}] WARNING: {exc}")

if not frames:
    raise RuntimeError(
        "No runs found. Check LABEL_DIRS paths and ensure grid-search runs exist."
    )

catalog = pd.concat(frames, ignore_index=True)

# Apply topology type filter
if FILTER_TOPOLOGY_TYPE and "topology_type" in catalog.columns:
    before = len(catalog)
    catalog = catalog[catalog["topology_type"] == FILTER_TOPOLOGY_TYPE].copy()
    print(f"Filtered to topology_type='{FILTER_TOPOLOGY_TYPE}': {before} -> {len(catalog)} runs")

catalog = catalog.reset_index(drop=True)

# Status
catalog["status"] = "Completed"
catalog.loc[catalog.get("completed", pd.Series(True, index=catalog.index)) == False, "status"] = "Timed out"
if "worker_oom_detected" in catalog.columns:
    catalog.loc[catalog["worker_oom_detected"] == True, "status"] = "OOM"

print(f"\nTotal: {len(catalog)} runs")
for lbl in catalog["run_label"].unique():
    sub = catalog[catalog["run_label"] == lbl]
    n_c = (sub["status"] == "Completed").sum()
    n_t = (sub["status"] == "Timed out").sum()
    n_o = (sub["status"] == "OOM").sum()
    print(f"  [{lbl}] {len(sub)} runs — Completed: {n_c}, Timed out: {n_t}, OOM: {n_o}")

vary_cols = detect_varying_params(catalog)
print(f"\nVarying parameters across all runs: {vary_cols}")

## Build Summary DataFrame

In [ ]:
import hashlib, pickle

def _fingerprint(cat: pd.DataFrame) -> str:
    parts = []
    for _, row in cat.iterrows():
        p = Path(row["profiler_dir"])
        mtime = max((f.stat().st_mtime for f in p.glob("*.csv")), default=0) if p.is_dir() else 0
        parts.append(f"{row['run_path']}:{mtime}")
    return hashlib.sha256("\n".join(sorted(parts)).encode()).hexdigest()[:16]

_cache_dir = Path(".cache")
_cache_dir.mkdir(exist_ok=True)
_fp = _fingerprint(catalog)
_cache_file = _cache_dir / f"label_comparison_{_fp}.pkl"

if _cache_file.exists():
    with open(_cache_file, "rb") as f:
        summary_df = pickle.load(f)
    print(f"Loaded from cache ({_cache_file.name})")
else:
    summary_df, _ = build_comparison_df(catalog, collect_ecall_distributions=False)
    # Re-attach run_label from catalog
    summary_df["run_label"] = catalog.loc[summary_df["_catalog_idx"].values, "run_label"].values
    summary_df["status"] = catalog.loc[summary_df["_catalog_idx"].values, "status"].values
    with open(_cache_file, "wb") as f:
        pickle.dump(summary_df, f)
    for old in _cache_dir.glob("label_comparison_*.pkl"):
        if old != _cache_file:
            old.unlink()
    print(f"Built and cached ({_cache_file.name})")

print(f"Summary: {len(summary_df)} runs")
for lbl in summary_df["run_label"].unique():
    print(f"  [{lbl}] {(summary_df['run_label'] == lbl).sum()} runs")

# Per-label short labels using varying params
summary_df["config_label"] = summary_df.apply(
    lambda r: short_label(r, vary_cols), axis=1
)

## Prepare Traffic DataFrame

In [ ]:
traffic_df = None

if "traffic_compliance_rate" not in summary_df.columns or summary_df["traffic_compliance_rate"].isna().all():
    print("No traffic compliance data available (no DP lifecycle events found in profiler CSVs).")
else:
    traffic_df = summary_df.dropna(subset=["traffic_compliance_rate"]).copy()

    if MIN_EPOCHS > 0 and "max_epoch_completed" in traffic_df.columns:
        traffic_df = traffic_df[traffic_df["max_epoch_completed"] >= MIN_EPOCHS]
        print(f"Filtered to {len(traffic_df)} runs with >= {MIN_EPOCHS} epochs completed")

    # Throughput columns
    _fwd = next(
        (c for c in traffic_df.columns if c.startswith("counter_") and "forwarded" in c),
        next((c for c in traffic_df.columns if c.startswith("counter_") and "real_emissions" in c), None)
    )
    if _fwd and "active_duration_s" in traffic_df.columns:
        traffic_df["throughput_tuples_per_s"] = traffic_df[_fwd] / traffic_df["active_duration_s"]
        traffic_df["tuples_per_epoch"] = traffic_df[_fwd] / traffic_df["max_epoch_completed"].clip(lower=1)
        print(f"Added throughput columns using '{_fwd}'")

    # Normalized tick irregularity
    if "traffic_tick_interval_s" in traffic_df.columns:
        traffic_df["norm_mean_delta"] = traffic_df["traffic_mean_delta_s"] / traffic_df["traffic_tick_interval_s"]
        traffic_df["norm_std_delta"]  = traffic_df["traffic_std_delta_s"]  / traffic_df["traffic_tick_interval_s"]
        if "traffic_max_deviation_s" in traffic_df.columns:
            traffic_df["norm_max_dev"] = traffic_df["traffic_max_deviation_s"] / traffic_df["traffic_tick_interval_s"]

    print(f"Traffic DataFrame: {len(traffic_df)} runs")
    for lbl in traffic_df["run_label"].unique():
        sub = traffic_df[traffic_df["run_label"] == lbl]
        print(f"  [{lbl}] n={len(sub)}  "
              f"compliance mean={sub['traffic_compliance_rate'].mean():.3f}  "
              f"std={sub['traffic_compliance_rate'].std():.3f}")

---
## Section 1: Traffic Compliance Comparison

### 1a — Compliance distribution per label

Box + strip plot of the traffic compliance rate, one box per label.
A higher compliance rate means the bolt respects the tick interval more consistently.

In [ ]:
if traffic_df is None or traffic_df.empty:
    print("Skipped: no traffic compliance data.")
else:
    labels_order = list(traffic_df["run_label"].unique())
    n_labels = len(labels_order)
    palette = [color_for(i) for i in range(n_labels)]

    fig, ax = plt.subplots(figsize=(max(6, n_labels * 2.5), 5))

    for i, lbl in enumerate(labels_order):
        sub = traffic_df[traffic_df["run_label"] == lbl]["traffic_compliance_rate"].dropna().values
        colour = palette[i]
        bp = ax.boxplot(
            sub, positions=[i], widths=0.4, patch_artist=True,
            boxprops=dict(facecolor=colour, alpha=0.35),
            medianprops=dict(color=colour, linewidth=2.5),
            whiskerprops=dict(color=colour), capprops=dict(color=colour),
            flierprops=dict(marker=""),
        )
        jitter = np.random.default_rng(42).uniform(-0.12, 0.12, len(sub))
        ax.scatter(np.full(len(sub), i) + jitter, sub,
                   c=colour, s=45, alpha=0.7, edgecolors="white", linewidths=0.5, zorder=3)

    ax.axhline(1.0, color="#27AE60", linestyle="--", linewidth=1, alpha=0.7, label="100% compliance")
    ax.axhline(0.95, color="#E67E22", linestyle=":", linewidth=1, alpha=0.7, label="95% threshold")
    ax.set_xticks(range(n_labels))
    ax.set_xticklabels(labels_order, fontsize=11)
    ax.set_ylabel("Traffic compliance rate")
    ax.set_ylim(0, 1.08)
    ax.set_title("Traffic Compliance Rate by Label", fontsize=13, fontweight="bold")
    ax.legend(fontsize=9)
    ax.grid(True, axis="y", alpha=0.25)
    fig.tight_layout()
    save_or_show(fig, OUTPUT_DIR if SAVE_PLOTS else None, "1a-compliance-by-label", PLOT_FORMAT, True)

### 1b — Per-run compliance strip chart (sorted)

Each run is a bar. Runs from different labels are coloured differently.
This lets you see if individual runs within a label cluster together or spread widely.

In [ ]:
if traffic_df is None or traffic_df.empty:
    print("Skipped.")
else:
    labels_order = list(traffic_df["run_label"].unique())
    palette = {lbl: color_for(i) for i, lbl in enumerate(labels_order)}

    plot_df = traffic_df.dropna(subset=["traffic_compliance_rate"]).sort_values("traffic_compliance_rate").copy()
    n = len(plot_df)

    fig, ax = plt.subplots(figsize=(10, max(4, n * 0.42)))
    colors = [palette[r] for r in plot_df["run_label"]]
    ax.barh(range(n), plot_df["traffic_compliance_rate"], color=colors, alpha=0.8, edgecolor="white")

    for i, (_, row) in enumerate(plot_df.iterrows()):
        v = row.get("traffic_violations", 0)
        total = row.get("traffic_total_emissions", 0)
        if pd.notna(v) and pd.notna(total):
            ax.text(row["traffic_compliance_rate"] + 0.005, i,
                    f"{int(v)}/{int(total)}", va="center", fontsize=7, color="#555")

    ax.set_yticks(range(n))
    ax.set_yticklabels(plot_df["config_label"], fontsize=7)
    ax.set_xlabel("Compliance rate")
    ax.set_xlim(0, 1.18)
    ax.axvline(1.0, color="#27AE60", linestyle="--", linewidth=1, alpha=0.6)
    ax.axvline(0.95, color="#E67E22", linestyle=":", linewidth=1, alpha=0.6)

    handles = [mpatches.Patch(color=palette[lbl], label=lbl) for lbl in labels_order]
    handles += [
        plt.Line2D([], [], color="#27AE60", linestyle="--", linewidth=1, label="100%"),
        plt.Line2D([], [], color="#E67E22", linestyle=":", linewidth=1, label="95%"),
    ]
    ax.legend(handles=handles, fontsize=8, loc="lower right")
    ax.set_title("Per-Run Traffic Compliance (sorted)", fontsize=11, fontweight="bold")
    ax.grid(True, axis="x", alpha=0.2)
    fig.tight_layout()
    save_or_show(fig, OUTPUT_DIR if SAVE_PLOTS else None, "1b-compliance-strip", PLOT_FORMAT, True)

### 1c — Compliance vs parameters (sensitivity)

For each parameter that varies across runs, show how compliance changes with label as a grouping.
This reveals whether the fix helps more under high vs low workload.

In [ ]:
if traffic_df is None or traffic_df.empty:
    print("Skipped.")
else:
    col = "traffic_compliance_rate"
    vary = [c for c in vary_cols if c in traffic_df.columns and traffic_df[c].nunique() > 1]

    if not vary:
        print("No varying parameters — all runs share the same configuration.")
    else:
        labels_order = list(traffic_df["run_label"].unique())
        palette = {lbl: color_for(i) for i, lbl in enumerate(labels_order)}

        fig, axes = plt.subplots(1, len(vary), figsize=(5 * len(vary), 5), squeeze=False)

        for pi, param in enumerate(vary):
            ax = axes[0, pi]
            for lbl in labels_order:
                sub = traffic_df[traffic_df["run_label"] == lbl].dropna(subset=[col, param])
                if sub.empty:
                    continue
                grouped = sub.groupby(param)[col].agg(["mean", "std"]).sort_index()
                x, y = grouped.index.values, grouped["mean"].values
                err = grouped["std"].fillna(0).values
                ax.plot(x, y, marker="o", color=palette[lbl], label=lbl, linewidth=2)
                ax.fill_between(x, y - err, y + err, color=palette[lbl], alpha=0.15)

            ax.set_xlabel(_PARAM_DISPLAY.get(param, param))
            ax.set_ylabel("Compliance rate" if pi == 0 else "")
            ax.set_ylim(0, 1.05)
            ax.axhline(1.0, color="#27AE60", linestyle="--", linewidth=0.8, alpha=0.5)
            ax.axhline(0.95, color="#E67E22", linestyle=":", linewidth=0.8, alpha=0.5)
            ax.grid(True, alpha=0.2)
            if pi == 0:
                ax.legend(fontsize=9)

        fig.suptitle("Traffic Compliance vs Parameters — by Label", fontsize=12, fontweight="bold")
        fig.tight_layout()
        save_or_show(fig, OUTPUT_DIR if SAVE_PLOTS else None, "1c-compliance-sensitivity", PLOT_FORMAT, True)

---
## Section 2: Tick Tuple Irregularity

Measures how regularly tick tuples arrive at the bolt, relative to the configured tick interval.

- **Normalized mean delta** — should be 1.0 (mean inter-emission ≈ tick interval)
- **Normalized std delta** — should be 0.0 (no jitter)
- **Normalized max deviation** — worst-case single-step deviation

### 2a — Irregularity distribution per label

In [ ]:
if traffic_df is None or traffic_df.empty or "norm_mean_delta" not in traffic_df.columns:
    print("Skipped: no tick irregularity data.")
else:
    labels_order = list(traffic_df["run_label"].unique())
    palette = {lbl: color_for(i) for i, lbl in enumerate(labels_order)}

    panels = [
        ("norm_mean_delta", "Normalized mean delta\n(ideal = 1.0)", 1.0),
        ("norm_std_delta",  "Normalized std delta\n(ideal = 0.0)", 0.0),
    ]
    if "norm_max_dev" in traffic_df.columns:
        panels.append(("norm_max_dev", "Normalized max deviation\n(ideal = 0.0)", None))

    fig, axes = plt.subplots(1, len(panels), figsize=(5 * len(panels), 6), squeeze=False)

    for pi, (col, ylabel, ref) in enumerate(panels):
        ax = axes[0, pi]
        positions = list(range(len(labels_order)))
        for i, lbl in enumerate(labels_order):
            sub = traffic_df[traffic_df["run_label"] == lbl][col].dropna().values
            colour = palette[lbl]
            ax.boxplot(
                sub, positions=[i], widths=0.4, patch_artist=True,
                boxprops=dict(facecolor=colour, alpha=0.3),
                medianprops=dict(color=colour, linewidth=2.5),
                whiskerprops=dict(color=colour), capprops=dict(color=colour),
                flierprops=dict(marker=""),
            )
            jitter = np.random.default_rng(42 + pi).uniform(-0.12, 0.12, len(sub))
            ax.scatter(np.full(len(sub), i) + jitter, sub,
                       c=colour, s=35, alpha=0.65, edgecolors="white", linewidths=0.5, zorder=3,
                       label=lbl if pi == 0 else None)

        if ref is not None:
            ax.axhline(ref, color="#27AE60", linestyle="--", linewidth=1, alpha=0.7)
        ax.set_xticks(range(len(labels_order)))
        ax.set_xticklabels(labels_order, fontsize=10)
        ax.set_ylabel(ylabel)
        ax.grid(True, axis="y", alpha=0.25)

    if len(labels_order) > 1:
        axes[0, 0].legend(fontsize=9)

    fig.suptitle("Tick Tuple Irregularity by Label", fontsize=13, fontweight="bold")
    fig.tight_layout()
    save_or_show(fig, OUTPUT_DIR if SAVE_PLOTS else None, "2a-tick-irregularity-by-label", PLOT_FORMAT, True)

### 2b — Tick irregularity vs parameters

In [ ]:
if traffic_df is None or traffic_df.empty or "norm_std_delta" not in traffic_df.columns:
    print("Skipped.")
else:
    vary = [c for c in vary_cols if c in traffic_df.columns and traffic_df[c].nunique() > 1]
    if not vary:
        print("No varying parameters.")
    else:
        labels_order = list(traffic_df["run_label"].unique())
        palette = {lbl: color_for(i) for i, lbl in enumerate(labels_order)}

        irr_metrics = [
            ("norm_std_delta",  "Norm. std delta (↓ better)"),
            ("norm_max_dev",    "Norm. max deviation (↓ better)"),
        ]
        irr_metrics = [(col, lbl) for col, lbl in irr_metrics if col in traffic_df.columns]

        for metric_col, metric_label in irr_metrics:
            fig, axes = plt.subplots(1, len(vary), figsize=(5 * len(vary), 5), squeeze=False)
            for pi, param in enumerate(vary):
                ax = axes[0, pi]
                for lbl in labels_order:
                    sub = traffic_df[traffic_df["run_label"] == lbl].dropna(subset=[metric_col, param])
                    if sub.empty:
                        continue
                    grouped = sub.groupby(param)[metric_col].agg(["mean", "std"]).sort_index()
                    x, y = grouped.index.values, grouped["mean"].values
                    err = grouped["std"].fillna(0).values
                    ax.plot(x, y, marker="o", color=palette[lbl], label=lbl, linewidth=2)
                    ax.fill_between(x, y - err, y + err, color=palette[lbl], alpha=0.15)
                ax.set_xlabel(_PARAM_DISPLAY.get(param, param))
                ax.set_ylabel(metric_label if pi == 0 else "")
                ax.grid(True, alpha=0.2)
                if pi == 0:
                    ax.legend(fontsize=9)
            fig.suptitle(f"{metric_label} vs Parameters — by Label", fontsize=12, fontweight="bold")
            fig.tight_layout()
            save_or_show(fig, OUTPUT_DIR if SAVE_PLOTS else None,
                         f"2b-irr-{metric_col}-sensitivity", PLOT_FORMAT, True)

---
## Section 3: Throughput Comparison

### 3a — Throughput distribution (tuples/s and tuples/epoch)

In [ ]:
if traffic_df is None or traffic_df.empty or "throughput_tuples_per_s" not in traffic_df.columns:
    print("Skipped: no throughput data.")
else:
    labels_order = list(traffic_df["run_label"].unique())
    palette = {lbl: color_for(i) for i, lbl in enumerate(labels_order)}

    tput_metrics = [
        ("throughput_tuples_per_s", "Throughput (tuples/s)"),
        ("tuples_per_epoch",        "Tuples per epoch"),
    ]
    tput_metrics = [(c, l) for c, l in tput_metrics if c in traffic_df.columns]

    fig, axes = plt.subplots(1, len(tput_metrics), figsize=(5 * len(tput_metrics), 5), squeeze=False)

    for pi, (col, ylabel) in enumerate(tput_metrics):
        ax = axes[0, pi]
        for i, lbl in enumerate(labels_order):
            sub = traffic_df[traffic_df["run_label"] == lbl][col].dropna().values
            colour = palette[lbl]
            ax.boxplot(
                sub, positions=[i], widths=0.4, patch_artist=True,
                boxprops=dict(facecolor=colour, alpha=0.3),
                medianprops=dict(color=colour, linewidth=2.5),
                whiskerprops=dict(color=colour), capprops=dict(color=colour),
                flierprops=dict(marker=""),
            )
            jitter = np.random.default_rng(pi * 7).uniform(-0.12, 0.12, len(sub))
            ax.scatter(np.full(len(sub), i) + jitter, sub,
                       c=colour, s=35, alpha=0.65, edgecolors="white", linewidths=0.5, zorder=3)

        ax.set_xticks(range(len(labels_order)))
        ax.set_xticklabels(labels_order, fontsize=10)
        ax.set_ylabel(ylabel)
        ax.grid(True, axis="y", alpha=0.25)

    fig.suptitle("Throughput by Label", fontsize=13, fontweight="bold")
    fig.tight_layout()
    save_or_show(fig, OUTPUT_DIR if SAVE_PLOTS else None, "3a-throughput-by-label", PLOT_FORMAT, True)

### 3b — Compliance vs throughput (does the fix decouple them?)

In the old design, higher throughput → worse compliance (the lock was the bottleneck).
After the fix, compliance should be **independent of throughput**.

In [ ]:
if (traffic_df is None or traffic_df.empty
        or "throughput_tuples_per_s" not in traffic_df.columns
        or "traffic_compliance_rate" not in traffic_df.columns):
    print("Skipped: missing throughput or compliance data.")
else:
    labels_order = list(traffic_df["run_label"].unique())
    palette = {lbl: color_for(i) for i, lbl in enumerate(labels_order)}

    fig, ax = plt.subplots(figsize=(8, 5))
    for lbl in labels_order:
        sub = traffic_df[traffic_df["run_label"] == lbl].dropna(
            subset=["throughput_tuples_per_s", "traffic_compliance_rate"]
        )
        ax.scatter(
            sub["throughput_tuples_per_s"], sub["traffic_compliance_rate"],
            c=palette[lbl], s=60, alpha=0.75, edgecolors="white", linewidths=0.5,
            label=lbl, zorder=3,
        )

    ax.axhline(1.0, color="#27AE60", linestyle="--", linewidth=1, alpha=0.6)
    ax.axhline(0.95, color="#E67E22", linestyle=":", linewidth=1, alpha=0.6)
    ax.set_xlabel("Throughput (tuples/s)")
    ax.set_ylabel("Traffic compliance rate")
    ax.set_ylim(0, 1.08)
    ax.set_title("Compliance vs Throughput — by Label", fontsize=12, fontweight="bold")
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.2)
    fig.tight_layout()
    save_or_show(fig, OUTPUT_DIR if SAVE_PLOTS else None, "3b-compliance-vs-throughput", PLOT_FORMAT, True)

---
## Section 4: Snapshot Duration

Snapshot duration measures the time spent inside the enclave running the DP algorithm
(Algorithm 1 + 2 + 3). The fix does **not** change the algorithm — snapshot duration
should remain the same. A change here would indicate a regression.

### 4a — Snapshot duration distribution by label

In [ ]:
snap_col = "avg_snapshot_duration_s"
snap_df = summary_df.dropna(subset=[snap_col]) if snap_col in summary_df.columns else pd.DataFrame()

if snap_df.empty:
    print("Skipped: no snapshot duration data.")
else:
    labels_order = list(snap_df["run_label"].unique())
    palette = {lbl: color_for(i) for i, lbl in enumerate(labels_order)}

    fig, ax = plt.subplots(figsize=(max(6, len(labels_order) * 2.5), 5))
    for i, lbl in enumerate(labels_order):
        sub = snap_df[snap_df["run_label"] == lbl][snap_col].dropna().values * 1000  # → ms
        colour = palette[lbl]
        ax.boxplot(
            sub, positions=[i], widths=0.4, patch_artist=True,
            boxprops=dict(facecolor=colour, alpha=0.3),
            medianprops=dict(color=colour, linewidth=2.5),
            whiskerprops=dict(color=colour), capprops=dict(color=colour),
            flierprops=dict(marker=""),
        )
        jitter = np.random.default_rng(99).uniform(-0.12, 0.12, len(sub))
        ax.scatter(np.full(len(sub), i) + jitter, sub,
                   c=colour, s=40, alpha=0.65, edgecolors="white", linewidths=0.5, zorder=3, label=lbl)

    ax.set_xticks(range(len(labels_order)))
    ax.set_xticklabels(labels_order, fontsize=10)
    ax.set_ylabel("Avg snapshot duration (ms)")
    ax.set_title("Snapshot Duration by Label", fontsize=12, fontweight="bold")
    ax.grid(True, axis="y", alpha=0.25)
    ax.legend(fontsize=9)
    fig.tight_layout()
    save_or_show(fig, OUTPUT_DIR if SAVE_PLOTS else None, "4a-snapshot-duration-by-label", PLOT_FORMAT, True)

### 4b — Epoch duration by label

Epoch duration = wall-clock time between consecutive `EPOCH_ADVANCED` events.
Should be approximately equal to the tick interval.
If the fix improved compliance, epoch duration variance should decrease.

In [ ]:
epoch_col = "avg_epoch_duration_s"
epoch_df = summary_df.dropna(subset=[epoch_col]) if epoch_col in summary_df.columns else pd.DataFrame()

if epoch_df.empty:
    print("Skipped: no epoch duration data.")
else:
    labels_order = list(epoch_df["run_label"].unique())
    palette = {lbl: color_for(i) for i, lbl in enumerate(labels_order)}

    fig, ax = plt.subplots(figsize=(max(6, len(labels_order) * 2.5), 5))
    for i, lbl in enumerate(labels_order):
        sub = epoch_df[epoch_df["run_label"] == lbl][epoch_col].dropna().values
        colour = palette[lbl]
        ax.boxplot(
            sub, positions=[i], widths=0.4, patch_artist=True,
            boxprops=dict(facecolor=colour, alpha=0.3),
            medianprops=dict(color=colour, linewidth=2.5),
            whiskerprops=dict(color=colour), capprops=dict(color=colour),
            flierprops=dict(marker=""),
        )
        jitter = np.random.default_rng(77).uniform(-0.12, 0.12, len(sub))
        ax.scatter(np.full(len(sub), i) + jitter, sub,
                   c=colour, s=40, alpha=0.65, edgecolors="white", linewidths=0.5, zorder=3)

    # Reference line: tick interval (if unique and available)
    if "tick_interval_secs" in epoch_df.columns and epoch_df["tick_interval_secs"].nunique() == 1:
        tick_s = epoch_df["tick_interval_secs"].iloc[0]
        ax.axhline(tick_s, color="#27AE60", linestyle="--", linewidth=1, alpha=0.7,
                   label=f"Tick interval ({tick_s}s)")
        ax.legend(fontsize=9)

    ax.set_xticks(range(len(labels_order)))
    ax.set_xticklabels(labels_order, fontsize=10)
    ax.set_ylabel("Avg epoch duration (s)")
    ax.set_title("Epoch Duration by Label", fontsize=12, fontweight="bold")
    ax.grid(True, axis="y", alpha=0.25)
    fig.tight_layout()
    save_or_show(fig, OUTPUT_DIR if SAVE_PLOTS else None, "4b-epoch-duration-by-label", PLOT_FORMAT, True)

---
## Section 5: Real vs Dummy Emission Ratio

Each tick the bolt emits either a **real partial** (snapshot ready) or a **dummy** (still computing).
More dummies = the background snapshot thread didn't finish in time for the tick.

The fix should **not** change this ratio directly — it only prevents the executor thread from
blocking during snapshot, not the snapshot duration itself.
If the ratio changes, it could indicate a side-effect of the change.

In [ ]:
real_col  = next((c for c in summary_df.columns if "real_emissions" in c and c.startswith("counter_")), None)
dummy_col = next((c for c in summary_df.columns if "dummy_emissions" in c and c.startswith("counter_")), None)

if real_col is None or dummy_col is None:
    print(f"Skipped: emission counters not found (real={real_col}, dummy={dummy_col}).")
else:
    ratio_df = summary_df.dropna(subset=[real_col, dummy_col]).copy()
    total = ratio_df[real_col] + ratio_df[dummy_col]
    ratio_df["real_ratio"] = ratio_df[real_col] / total.clip(lower=1)

    labels_order = list(ratio_df["run_label"].unique())
    palette = {lbl: color_for(i) for i, lbl in enumerate(labels_order)}

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))

    # Left: real ratio distribution
    ax = axes[0]
    for i, lbl in enumerate(labels_order):
        sub = ratio_df[ratio_df["run_label"] == lbl]["real_ratio"].dropna().values
        colour = palette[lbl]
        ax.boxplot(sub, positions=[i], widths=0.4, patch_artist=True,
                   boxprops=dict(facecolor=colour, alpha=0.3),
                   medianprops=dict(color=colour, linewidth=2.5),
                   whiskerprops=dict(color=colour), capprops=dict(color=colour),
                   flierprops=dict(marker=""))
        jitter = np.random.default_rng(11).uniform(-0.12, 0.12, len(sub))
        ax.scatter(np.full(len(sub), i) + jitter, sub,
                   c=colour, s=40, alpha=0.65, edgecolors="white", linewidths=0.5, zorder=3)
    ax.set_xticks(range(len(labels_order)))
    ax.set_xticklabels(labels_order, fontsize=10)
    ax.set_ylabel("Real emission fraction")
    ax.set_ylim(0, 1.08)
    ax.set_title("Real / (Real + Dummy) Ratio", fontsize=11)
    ax.grid(True, axis="y", alpha=0.25)

    # Right: stacked bar — mean real vs dummy per label
    ax2 = axes[1]
    means = ratio_df.groupby("run_label")[[real_col, dummy_col]].mean().reindex(labels_order)
    x = np.arange(len(labels_order))
    ax2.bar(x, means[real_col],  color=[palette[l] for l in labels_order], alpha=0.8, label="Real")
    ax2.bar(x, means[dummy_col], bottom=means[real_col],
            color=[palette[l] for l in labels_order], alpha=0.35, hatch="//", label="Dummy")
    ax2.set_xticks(x)
    ax2.set_xticklabels(labels_order, fontsize=10)
    ax2.set_ylabel("Mean emission count")
    ax2.set_title("Mean Real vs Dummy Emissions per Run", fontsize=11)
    ax2.legend(fontsize=9)
    ax2.grid(True, axis="y", alpha=0.25)

    fig.suptitle("Real vs Dummy Emission Ratio — by Label", fontsize=13, fontweight="bold")
    fig.tight_layout()
    save_or_show(fig, OUTPUT_DIR if SAVE_PLOTS else None, "5-real-dummy-ratio", PLOT_FORMAT, True)

---
## Section 6: Summary Table

Key metrics grouped by label, averaged across all matching runs.

In [ ]:
wanted = [
    "run_label",
    "traffic_compliance_rate",
    "traffic_violations",
    "traffic_total_emissions",
    "norm_std_delta",
    "norm_mean_delta",
    "norm_max_dev",
    "throughput_tuples_per_s",
    "tuples_per_epoch",
    "avg_snapshot_duration_s",
    "avg_epoch_duration_s",
]

# Use traffic_df if available (has norm_* columns), else fall back to summary_df
_base = traffic_df if traffic_df is not None and not traffic_df.empty else summary_df
avail = [c for c in wanted if c in _base.columns]

agg = (
    _base[avail]
    .groupby("run_label")
    .agg(["mean", "std", "count"])
)

display(agg.round(4))

In [ ]:
# Flat summary: one row per label, mean ± std for key metrics
_base = traffic_df if traffic_df is not None and not traffic_df.empty else summary_df
key_metrics = [
    ("traffic_compliance_rate",  "Compliance rate"),
    ("norm_std_delta",           "Norm. std delta"),
    ("norm_max_dev",             "Norm. max deviation"),
    ("throughput_tuples_per_s",  "Throughput (t/s)"),
    ("avg_snapshot_duration_s",  "Avg snapshot (s)"),
    ("avg_epoch_duration_s",     "Avg epoch (s)"),
]

rows = []
for lbl in _base["run_label"].unique():
    sub = _base[_base["run_label"] == lbl]
    row = {"Label": lbl, "N runs": len(sub)}
    for col, display_name in key_metrics:
        if col in sub.columns:
            vals = sub[col].dropna()
            row[display_name] = f"{vals.mean():.4f} ± {vals.std():.4f}" if len(vals) > 0 else "—"
        else:
            row[display_name] = "—"
    rows.append(row)

display(pd.DataFrame(rows).set_index("Label"))